In [638]:
import numpy as np
import gymnasium as gym
from gymnasium.utils.env_checker import check_env

In [639]:
def simulate_tracks(
    n_planes,
    spatial_resolution,
    n_tracks=1000,
    plane_size=100.0,
    start_area_fraction=0.10,
    z_min=0.0,
    z_max=600.0,
    theta_max_deg=10.0,
    beta_min=0.5,
    beta_max=0.75,
    time_resolution=2.0,
    tof_margin=50.0,
    default_value=-999999.0,
    seed=42
):
    """
    Simula tracce Monte Carlo con:
    - n_planes piani di tracciamento sensibili a x,y
    - 2 piani temporali, uno prima e uno dopo il tracciatore
    - velocità casuale tra beta_min*c e beta_max*c
    - controllo dell'accettanza geometrica dei piani

    Le hit fuori accettanza vengono riempite con default_value.

    Returns
    -------
    x_meas : np.ndarray
        Coordinate x misurate, shape = (n_tracks, n_planes).
        Se fuori accettanza: default_value.

    y_meas : np.ndarray
        Coordinate y misurate, shape = (n_tracks, n_planes).
        Se fuori accettanza: default_value.

    z_planes : np.ndarray
        Coordinate z dei piani di tracciamento, shape = (n_planes,)

    theta : np.ndarray
        Angolo polare generato rispetto all'asse z, in radianti.

    theta_deg : np.ndarray
        Angolo polare generato rispetto all'asse z, in gradi.

    beta : np.ndarray
        Velocità generata in unità di c.

    delta_t_meas : np.ndarray
        Differenza di tempo misurata tra i due piani temporali, in ns.

    hit_valid : np.ndarray
        Maschera booleana, shape = (n_tracks, n_planes).
        True se la hit è dentro l'accettanza del piano.
    """

    rng = np.random.default_rng(seed)

    # Velocità della luce in mm/ns
    c_mm_ns = 299.792458

    # Posizione dei piani di tracciamento
    z_planes = np.linspace(z_min, z_max, n_planes)

    # Posizione dei due piani temporali
    z_time_1 = z_min - tof_margin
    z_time_2 = z_max + tof_margin

    delta_z_tof = z_time_2 - z_time_1

    # Regione iniziale pari a start_area_fraction dell'area del piano
    start_size = plane_size * np.sqrt(start_area_fraction)

    start_half_size = start_size / 2

    x0 = rng.uniform(-start_half_size, start_half_size, n_tracks)
    y0 = rng.uniform(-start_half_size, start_half_size, n_tracks)

    # Angoli casuali entro un cono attorno all'asse z
    theta_max = np.deg2rad(theta_max_deg)

    phi = rng.uniform(0, 2 * np.pi, n_tracks)

    # Distribuzione uniforme in angolo solido
    cos_theta = rng.uniform(np.cos(theta_max), 1, n_tracks)
    theta = np.arccos(cos_theta)
    theta_deg = np.rad2deg(theta)

    tan_theta = np.tan(theta)

    tx = tan_theta * np.cos(phi)   # dx/dz
    ty = tan_theta * np.sin(phi)   # dy/dz

    # Velocità casuale
    beta = rng.uniform(beta_min, beta_max, n_tracks)

    # Hit vere sui piani di tracciamento
    x_true = x0[:, None] + tx[:, None] * z_planes[None, :]
    y_true = y0[:, None] + ty[:, None] * z_planes[None, :]

    # Maschera di accettanza geometrica
    half_plane_size = plane_size / 2

    hit_valid = (
        (np.abs(x_true) <= half_plane_size) &
        (np.abs(y_true) <= half_plane_size)
    )

    # Hit misurate con risoluzione spaziale gaussiana
    x_meas = x_true + rng.normal(
        0,
        spatial_resolution,
        size=x_true.shape
    )

    y_meas = y_true + rng.normal(
        0,
        spatial_resolution,
        size=y_true.shape
    )

    # Se la traccia è fuori accettanza, il piano non misura nessuna hit
    x_meas[~hit_valid] = default_value
    y_meas[~hit_valid] = default_value

    # Tempo di volo vero tra i due piani temporali
    path_length = delta_z_tof / np.cos(theta)
    delta_t_true = path_length / (beta * c_mm_ns)

    # Misura del tempo sui due piani temporali
    t1_meas = rng.normal(0, time_resolution, n_tracks)
    t2_meas = delta_t_true + rng.normal(0, time_resolution, n_tracks)

    delta_t_meas = t2_meas - t1_meas

    return x_meas, y_meas, z_planes, theta, theta_deg, beta, delta_t_meas, hit_valid

In [640]:
import numpy as np
from gymnasium import spaces

class DetectorOptimizerEnv(gym.Env):
    def __init__(self):
        super().__init__()

        # --- SPAZIO DELLE AZIONI ---
        # [n_piani, risoluzione_spaziale, risoluzione_temporale, lunghezza_totale_z]
        # n_piani: da 3 a 20
        # risoluzione_spaziale: da 1 a 10.0 mm
        # risoluzione_temporale: da 0.1 a 5.0 ns
        # lunghezza_totale (z_max): da 100 a 1000 mm
        self.n_tracks = 1

        self.n_planes = spaces.Box(low=2, high=20, dtype=np.int32)
        self.spatial_res = spaces.Box(low=100, high=1000, dtype=np.float32)
        self.temporal_res = spaces.Box(low=0.1, high=5, dtype=np.float32)
        self.z_max = spaces.Box(low=100, high=1000, dtype=np.float32)

        self.action_space = spaces.Dict({'n_planes': self.n_planes,'spatial_res': self.spatial_res,
                                          'temporal_res': self.temporal_res, 'z_max': self.z_max})


        self.x_meas = spaces.Box(low=-np.inf, high=np.inf, dtype=np.float32)
        self.y_meas = spaces.Box(low=-np.inf, high=np.inf, dtype=np.float32)
        #self.th = spaces.Box(low = 0, high=2*np.pi, dtype=np.float32)
        self.delta_meas = spaces.Box(low=-np.inf, high=np.inf, dtype=np.float32)
        self.hit_valid = spaces.MultiBinary(self.n_tracks)
        self.observation_space = spaces.Dict({'x_meas' : self.x_meas, 'y_meas': self.y_meas,'n_planes': self.n_planes, 
                                               'delta_meas': self.delta_meas, 'spatial_res': self.spatial_res,
                                               'temporal_res': self.temporal_res, 'z_max' :self.z_max, "hit_valid" : self.hit_valid})

        self.state = self.action_space.sample()

    def step(self, action):
        n_planes = action['n_planes']
        spatial_res = action['spatial_res']
        time_res = action['temporal_res']
        z_max_val = action['z_max']

        results = simulate_tracks(n_planes=int(n_planes[0]), spatial_resolution=spatial_res,
                                  time_resolution=time_res,z_max=z_max_val[0], n_tracks = self.n_tracks)
        beta_t = results[5]
        theta_t = results[3]
        
        cost = (n_planes * 1) + (spatial_res ** 1.5) + (time_res * 1)
        evals = self.__eval_sim__(x_meas = results[0], y_meas = results[1], z_planes= results[2],
                                          delta_t_meas = results[6], hit_valid=results[7])
        
        theta_dist = np.linalg.norm(theta_t - evals['theta_measured'])
        beta_dist = np.linalg.norm(beta_t-evals['beta_measured'])
        eta = 0.1
        reward = eta*-cost - (1-eta)*(theta_dist + beta_dist)
        self.state = action

        return self.__get_obs__(), reward, False, False, {"beta_dist": beta_dist, "theta_dist": theta_dist}
    
    def __eval_sim__(self, x_meas, y_meas, z_planes, delta_t_meas, hit_valid):
        # Indici del primo e dell'ultimo piano
       # Indici dei piani per il calcolo (Primo e Ultimo)
        i_start, i_end = 0, -1

        # --- CALCOLO GEOMETRICO ---
        dx = x_meas[:, i_end] - x_meas[:, i_start]
        dy = y_meas[:, i_end] - y_meas[:, i_start]
        dz = z_planes[i_end] - z_planes[i_start]
        
        # Distanza trasversale (proiezione sul piano XY)
        dr = np.sqrt(dx**2 + dy**2)
        
        # Distanza totale percorsa (3D)
        path_length = np.sqrt(dr**2 + dz**2)

        # --- STIMA VELOCITÀ (BETA) ---
        c = 29.9792458 # cm/ns
        with np.errstate(divide='ignore', invalid='ignore'):
            beta_measured = path_length / (c * delta_t_meas)

        # --- STIMA ANGOLO (THETA) ---
        # theta = arctan(opposto / adiacente) -> arctan(dr / dz)
        theta_rad_est = np.arctan2(dr, dz)
        theta_deg_est = np.degrees(theta_rad_est)

        # --- FILTRO VALIDITÀ ---
        # Escludiamo le tracce che non hanno colpito entrambi i piani di riferimento
        mask = hit_valid[:, i_start] & hit_valid[:, i_end]
        
        # Puliamo i dati impostando a NaN dove non validi
        beta_measured[~mask] = -100
        theta_deg_est[~mask] = -100

        return {"beta_measured":beta_measured, "theta_measured": theta_deg_est}

    def __get_obs__(self):
        result = simulate_tracks(n_planes = int(self.state['n_planes'][0]), spatial_resolution=self.state['spatial_res'],
                                time_resolution = self.state['temporal_res'], z_max = self.state['z_max'][0], n_tracks=self.n_tracks)

        return {"x_meas": result[0], "y_meas": result[1], "n_planes": self.state['n_planes'],
                "delta_meas": result[4], "spatial_res":self.state['spatial_res'], "temporal_res":self.state['temporal_res'],
                "z_max": self.state['z_max'][0], "hit_valid": result[7]}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.action_space.sample()
        return self.__get_obs__(), {}

In [641]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class DetectorActionWrapper(gym.ActionWrapper):
    def __init__(self, env):
        super().__init__(env)
        # Definiamo un unico spazio Box per SB3
        # Indici: 0=n_planes, 1=spatial_res, 2=temporal_res, 3=z_max
        low = np.array([2, 100.0, 0.1, 100.0], dtype=np.float32)
        high = np.array([19, 1000.0, 5.0, 1000.0], dtype=np.float32)
        
        self.action_space = spaces.Box(low=low, high=high, dtype=np.float32)

    def action(self, action):
        """
        Converte l'array di SB3 nel formato 'flat' che il tuo step() si aspetta,
        assicurandosi che i tipi siano corretti (es. n_planes deve essere int).
        """
        # action è un array numpy: [piani, spat, temp, z]
        n_planes = int(np.round(action[0]))
        spatial_res = float(action[1])
        temporal_res = float(action[2])
        z_max = float(action[3])
        
        # Restituiamo un array o una lista che il tuo step() possa indicizzare
        return [n_planes, spatial_res, temporal_res, [z_max]]

In [642]:
from torchrl.envs.libs.gym import GymWrapper

env = DetectorOptimizerEnv()
env = DetectorActionWrapper(env)



In [643]:
from stable_baselines3 import A2C, SAC, PPO

model = SAC('MultiInputPolicy', env, verbose=1)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [644]:
model.learn(total_timesteps=10)

ValueError: could not broadcast input array from shape (6,) into shape (1,)